# Lab 02: IPO Research Agent

## Business Context

The data is ready — filings are parsed, chunked, and indexed; stock performance is in a Delta table. Now build the research agent: the first version that can answer **"What did Snowflake say about competition, and how did SNOW perform?"**

This is the prototype the analysts will test. It combines unstructured text retrieval (Vector Search) with structured data lookup (UC SQL functions) in a single agent.

| Attribute | Detail |
|---|---|
| **Exam Domain** | Application Development (30%) |
| **Key Skills** | @tool decorator, UC SQL functions, UCFunctionToolkit, create_react_agent, multi-tool agents |
| **Estimated Time** | 30 minutes |
| **Estimated Cost** | ~$2-3 |

In [ ]:
%pip install databricks-sdk databricks-vectorsearch mlflow langchain langchain-core langchain-community langgraph unitycatalog-ai[databricks] unitycatalog-langchain --quiet
dbutils.library.restartPython()

In [ ]:
CATALOG = "ipo_analyzer"
SCHEMA = "default"
LLM_ENDPOINT = "databricks-llama-4-maverick"
VS_ENDPOINT = "ipo_analyzer_vs_endpoint"
VS_INDEX = f"{CATALOG}.{SCHEMA}.filing_chunks_index"

print(f"Catalog : {CATALOG}")
print(f"Schema  : {SCHEMA}")
print(f"LLM     : {LLM_ENDPOINT}")

## A. Build the Retrieval Tool

The retrieval tool wraps Vector Search — when the agent needs to find relevant passages from S-1 filings, it calls this tool. The `@tool` decorator turns any Python function into a tool the LLM can invoke.

`query_type="HYBRID"` combines semantic search (meaning) with keyword search (exact terms) via Reciprocal Rank Fusion — the recommended default for RAG.

In [ ]:
from databricks.vector_search.client import VectorSearchClient
from langchain_core.tools import tool

vsc = VectorSearchClient()
index = vsc.get_index(VS_ENDPOINT, VS_INDEX)


def retrieve_context(query: str, num_results: int = 5) -> str:
    """Query the Vector Search index and return formatted context passages."""
    results = index.similarity_search(
        query_text=query,
        columns=["chunk_text", "path"],
        num_results=num_results,
        query_type="HYBRID",
    )
    docs = results.get("result", {}).get("data_array", [])
    parts = []
    for doc in docs:
        source = doc[1].split("/")[-1].replace("-S1.html", "").replace(".html", "")
        parts.append(f"[Source: {source}]\n{doc[0]}")
    return "\n\n---\n\n".join(parts) if parts else "No relevant passages found."


@tool
def search_filings(query: str) -> str:
    """Search S-1 filing text for passages relevant to the query.
    Use this for questions about what companies said in their IPO filings."""
    return retrieve_context(query)


# Smoke test
print(search_filings.invoke("Snowflake competitive advantages")[:500])

## B. Create UC Functions as Tools

Unity Catalog functions are governed, reusable SQL or Python functions stored in the catalog. When exposed as agent tools via `UCFunctionToolkit`, the LLM can call them like any other tool.

Two functions:
- **`get_filing_metadata`** — How many chunks do we have for a given company? (queries the chunks table)
- **`get_stock_performance`** — What was the stock's 12-month return? (queries the stock table)

> **Why SQL?** These are simple lookups over Delta tables. SQL UDFs have no cold-start overhead and are natively optimized by Spark. The `COMMENT` field becomes the tool description the LLM reads to decide when to call it.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_filing_metadata(company_name STRING)
RETURNS TABLE (path STRING, chunk_count BIGINT)
COMMENT 'Return the filing path and total chunk count for a company. Use this to check what filings are available and how much content each contains. Search by company name (e.g., Snowflake, DoorDash).'
RETURN
  SELECT path, COUNT(*) AS chunk_count
  FROM {CATALOG}.{SCHEMA}.filing_chunks
  WHERE LOWER(path) LIKE CONCAT('%', LOWER(company_name), '%')
  GROUP BY path
  ORDER BY chunk_count DESC
""")

print(f"Created: {CATALOG}.{SCHEMA}.get_filing_metadata")
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.get_filing_metadata('snowflake')"))

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_stock_performance(ticker_symbol STRING)
RETURNS TABLE (company STRING, ticker STRING, sector STRING, ipo_date STRING,
               ipo_price DOUBLE, price_3m DOUBLE, price_6m DOUBLE, price_12m DOUBLE,
               twelve_month_return_pct DOUBLE)
COMMENT 'Look up post-IPO stock performance for a company by ticker symbol (e.g., SNOW, DASH, COIN). Returns IPO price, prices at 3/6/12 months, and percentage return.'
RETURN
  SELECT company, ticker, sector, ipo_date, ipo_price, price_3m, price_6m, price_12m, twelve_month_return_pct
  FROM {CATALOG}.{SCHEMA}.stock_performance
  WHERE UPPER(ticker) = UPPER(ticker_symbol)
""")

print(f"Created: {CATALOG}.{SCHEMA}.get_stock_performance")
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.get_stock_performance('SNOW')"))

# ── Python UDF alternative (equivalent but with cold-start overhead): ─────
# spark.sql(f"""
# CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_stock_performance(ticker_symbol STRING)
# RETURNS STRING
# LANGUAGE PYTHON
# COMMENT 'Look up post-IPO stock performance by ticker.'
# AS $$
#     # Python UDFs have ~1-2s cold start. SQL is preferred for simple lookups.
#     return f"Stock lookup for {{ticker_symbol}}"
# $$
# """)

## C. Build the Multi-Tool Agent

Now we combine all three tools in a single **LangGraph ReAct agent**:

1. **`search_filings`** — Vector Search retrieval (from Section A)
2. **`get_filing_metadata`** — UC SQL function via `UCFunctionToolkit`
3. **`get_stock_performance`** — UC SQL function via `UCFunctionToolkit`

`UCFunctionToolkit` reads the UC function schemas (name, `COMMENT`, parameter types) and builds LangChain-compatible tool objects automatically. The LLM selects the appropriate tool(s) based on the question.

`create_react_agent` (LangGraph) manages the reasoning loop: the LLM decides which tool to call, LangGraph executes it, feeds the result back, and repeats until the LLM has enough information to answer.

In [ ]:
from langchain_community.chat_models import ChatDatabricks
from langgraph.prebuilt import create_react_agent
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
from unitycatalog.ai.langchain.toolkit import UCFunctionToolkit

# ── LLM ──────────────────────────────────────────────────────────────────
llm = ChatDatabricks(endpoint=LLM_ENDPOINT, max_tokens=1024, temperature=0.1)

# ── UC function tools ────────────────────────────────────────────────────
client = DatabricksFunctionClient()
uc_toolkit = UCFunctionToolkit(
    function_names=[
        f"{CATALOG}.{SCHEMA}.get_filing_metadata",
        f"{CATALOG}.{SCHEMA}.get_stock_performance",
    ],
    client=client,
)

# ── Combine all tools ────────────────────────────────────────────────────
all_tools = [search_filings] + uc_toolkit.tools
print(f"Tools: {[t.name for t in all_tools]}")

# ── System prompt ────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are an IPO Filing Analyzer for a financial research firm. "
    "You have access to S-1 filings from tech IPOs and stock performance data.\n\n"
    "- search_filings: Search S-1 text for relevant passages\n"
    "- get_filing_metadata: Look up chunk count and filing info for a company\n"
    "- get_stock_performance: Look up stock returns by ticker symbol\n\n"
    "Always cite the source filing when answering research questions. "
    "When asked about stock performance, use get_stock_performance with the ticker.\n\n"
    "IMPORTANT: You provide financial ANALYSIS, not investment ADVICE. "
    "Never recommend buying or selling stocks."
)

# ── Agent ────────────────────────────────────────────────────────────────
agent = create_react_agent(llm, all_tools, prompt=SYSTEM_PROMPT)
print("Agent ready.")

In [ ]:
# THE BUSINESS QUERY: combines filing text + stock data
query = "What did Snowflake say about competition in their S-1, and how did SNOW perform in its first year?"

print(f"Q: {query}")
print("=" * 70)
result = agent.invoke({"messages": [{"role": "user", "content": query}]})
print(result["messages"][-1].content)

In [ ]:
query2 = "Compare what Coinbase and Robinhood say about regulatory risk in their S-1 filings."

print(f"Q: {query2}")
print("=" * 70)
result2 = agent.invoke({"messages": [{"role": "user", "content": query2}]})
print(result2["messages"][-1].content)

In [ ]:
query3 = "How many chunks do we have for DoorDash, and what was DASH's 12-month stock return?"

print(f"Q: {query3}")
print("=" * 70)
result3 = agent.invoke({"messages": [{"role": "user", "content": query3}]})
print(result3["messages"][-1].content)

## Before / After

**Before this lab:** You had searchable chunks and a stock table, but no way to ask questions that span both. Getting "what did Snowflake say about competition AND how did the stock do?" required manually querying two separate systems.

**After this lab:** One agent, one question, one answer — grounded in the S-1 filing with citations, enriched with stock performance data.

In [ ]:
# BEFORE: Raw passages without synthesis
print("=== BEFORE: Raw passages (no agent) ===")
raw = retrieve_context("Snowflake competition", num_results=2)
print(raw[:400])
print("\n(Raw passages only — no synthesis, no stock data)\n")

# AFTER: Agent-synthesised answer with citations + stock data
print("=== AFTER: Agent answer ===")
result = agent.invoke({"messages": [{"role": "user", "content": "Briefly: what does Snowflake say about competition and how did SNOW perform?"}]})
print(result["messages"][-1].content)

In [ ]:
# Running scorecard — tracks cumulative progress across labs
from shared.lab_utils import get_scorecard
get_scorecard()

---

## Exam Preparation

### Key Concepts

| Concept | Definition |
|---|---|
| **`@tool` decorator** | Turns a Python function into a LangChain tool the LLM can invoke; the docstring becomes the tool description |
| **UC SQL Function** | A function stored in Unity Catalog, written in SQL, governed by UC permissions; `COMMENT` becomes tool description for LLMs |
| **UCFunctionToolkit** | LangChain integration that reads UC function schemas and converts them into LangChain `Tool` objects automatically |
| **`create_react_agent`** | LangGraph function that creates a ReAct agent — the LLM decides which tool to call, LangGraph manages the loop |
| **ReAct pattern** | Reasoning + Acting — the LLM reasons about what tool to call, acts by calling it, observes the result, and repeats |
| **Tool schema** | The name, description, and parameter types exposed to the LLM; sourced from the function name, `COMMENT`, and type annotations |
| **Multi-tool agent** | An agent with access to multiple tools; the LLM selects the right tool(s) per query based on tool descriptions |

### Practice Questions

**Q1.** How does an LLM know when to call a UC function that has been added as an agent tool?

- A) The agent hardcodes a routing table
- B) The LLM reads the function's name and `COMMENT` from the tool schema
- C) The agent calls every available tool and picks the best response
- D) The LLM relies solely on few-shot examples in the prompt

**Answer: B** — The LLM uses the tool schema (name + COMMENT description) to decide which tool matches the user's intent.

---

**Q2.** What is the correct way to create a LangGraph ReAct agent?

- A) `AgentExecutor.from_agent_and_tools(agent, tools)`
- B) `create_react_agent(llm, tools, prompt=system_prompt)`
- C) `llm.bind_tools(tools).invoke(messages)`
- D) `RunnableSequence(llm, tools)`

**Answer: B** — `create_react_agent` from `langgraph.prebuilt` is the current API. `AgentExecutor` is deprecated.

---

**Q3.** When should you prefer a UC function over a `@tool` for agent tool logic?

- A) When the logic makes outbound HTTP calls
- B) When the tool must return streaming responses
- C) When the logic queries Delta tables and needs UC governance (permissions, lineage)
- D) When you need sub-second latency

**Answer: C** — UC functions are governed, versioned, and optimized for Lakehouse queries. `@tool` is better for ephemeral or external API logic.

---

**Q4.** What does `query_type="HYBRID"` do in a Vector Search call?

- A) Searches using only keyword matching (BM25)
- B) Combines semantic search with keyword search using Reciprocal Rank Fusion
- C) Searches using only embedding similarity
- D) Searches across multiple indexes simultaneously

**Answer: B** — Hybrid combines the recall of semantic search with the precision of BM25 keyword matching.

---

**Q5.** A user asks "How did SNOW perform?" The agent has tools: search_filings, get_filing_metadata, get_stock_performance. Which tool should it call?

- A) search_filings — because SNOW might be mentioned in filings
- B) get_filing_metadata — because it looks up company info
- C) get_stock_performance — because the question is about stock performance
- D) All three tools in sequence

**Answer: C** — The question is specifically about stock performance, not filing content or metadata. The `COMMENT` on `get_stock_performance` says "Look up stock returns by ticker symbol."

### Cost Breakdown

| Component | Detail | Est. Cost |
|---|---|---|
| Serverless compute | Notebook execution (~20 min) | ~$0.50 |
| LLM tokens | Agent queries (3 test + 1 demo) | ~$1.00 |
| Vector Search | Retrieval calls during agent queries | ~$0.50 |
| **Total** | | **~$2-3** |